# FA15_SMOKE — F-A1.5, project verification (not part of the report; board R204)

**Plan:** `docs/FUTURE_RESEARCH_PLAN.md` §12 15/11 12:40. **Does:** ASR exactly as V9-X6 (asrpy 0.0.8, cutoff 20, per-file
calibration, one process per file) on the pipeline-filtered signal, then the record's statistics and features on the
cleaned signal; gates G-conv, G-lab, G-asr, G-ident and the projection of the data notebooks.
**Cases:** CHB-MIT ['chb17']; external ['PN00']; ASR only: chb06:chb06_01.edf, chb24:chb24_01.edf.
**Accelerator:** None (CPU). **Internet:** ON. **Inputs:** `bonesclarke26/combined-siena-scalp-and-chb-mit-eeg-datasets`, `thesis-r2-chbmit-features` (nhn2mm),
`thesis-r2-external-features` (norncreades).
**Runtime:** ≈ 1–2 h (computed: chb17 21 h + PN00 3.2 h + 5 h of ASR-only files at the laptop's 135 s per hour of EEG on 3 processes, up to 3 × on Kaggle, + features ≈ 0.3 h). Code: `github.com/Nhanvas/Thesis` at `6c99bab`; `src/fa15_asr.py` sha256 `132d6179a1c77723` (checked).
A gate failure is recorded; the END cell archives the results, then raises `STOPPED: ...`.
Do not edit; *Save Version → Save & Run All (Commit)*.

In [ ]:
# SETUP: packages + repository at the pinned commit (read only)
import os, sys, subprocess, time, json, hashlib, shutil
T_NB = time.time()
os.environ["FA15_T0"] = os.environ["R2_T0"] = str(T_NB)
NOTEBOOK, REPO_COMMIT, TAG = "FA15_SMOKE", "6c99bab5eb54391edb6fa5beff5c2a34aa3f442b", "smoke"
TEST = bool(os.environ.get("FA15_TEST_REPO"))                    # synthetic test in the build container only
WORK = os.environ.get("FA15_TEST_WORK", "/kaggle/working")
INPUT = os.environ.get("FA15_TEST_INPUT", "/kaggle/input")
SCRATCH = os.environ.get("FA15_TEST_SCRATCH", "/tmp/fa15")
for v in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS"):
    os.environ[v] = "1"
if TEST:
    REPO = os.environ["FA15_TEST_REPO"]
else:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "asrpy==0.0.8", "mne==1.12.1", "torch_geometric==2.7.0", "ruptures==1.1.10", "timescoring==0.0.7"])
    REPO = "/tmp/Thesis"
    if not os.path.exists(REPO + "/src/fa15_asr.py"):
        r = subprocess.run(f"git init -q {REPO} && cd {REPO} && git remote add origin https://github.com/Nhanvas/Thesis.git "
                           f"&& git fetch -q --depth 1 origin {REPO_COMMIT} && git checkout -q FETCH_HEAD", shell=True)
        if r.returncode != 0:
            subprocess.check_call(f"rm -rf {REPO} && git clone -q https://github.com/Nhanvas/Thesis.git {REPO} "
                                  f"&& cd {REPO} && git checkout -q {REPO_COMMIT}", shell=True)
    head = subprocess.check_output(["git", "-C", REPO, "rev-parse", "HEAD"]).decode().strip()
    assert head == REPO_COMMIT, f"wrong commit {head}"
sha = hashlib.sha256(open(REPO + "/src/fa15_asr.py", "rb").read()).hexdigest()[:16]
assert TEST or sha == "132d6179a1c77723", f"fa15_asr.py sha {sha} != 132d6179a1c77723"
os.environ.update(THESIS_REPO=REPO, FA15_WORK=WORK, FA15_SCRATCH=SCRATCH, R2_WORK=WORK, R2_INPUT=INPUT,
                  R2_SCRATCH=SCRATCH + "/r2")
sys.path.insert(0, REPO + "/src")
print("repository", REPO, "CPU cores", os.cpu_count(), f"({time.time()-T_NB:.0f}s)")

In [ ]:
# RUN: inventory (every raw file sha256 = PhysioNet), ASR per file (one process each), statistics + features per case
# (record code), gates; results in WORK/fa15/smoke/, datasets in WORK/fa15-asr-*-features-smoke/
import fa15_asr as F
CHB, EXT, ASR_ONLY = ["chb17"], ["PN00"], ["chb06:chb06_01.edf", "chb24:chb24_01.edf"]
if TEST and os.environ.get("FA15_TEST_CASES"):                   # the synthetic world holds a few cases only
    CHB, EXT, ASR_ONLY = json.loads(os.environ["FA15_TEST_CASES"])
FAILED = F.run(TAG, CHB, EXT, ASR_ONLY, smoke=True, notebook=NOTEBOOK, commit=REPO_COMMIT)
print("failed gates:", FAILED or "none")

In [ ]:
# END: archive the results (small files), measured runtime; datasets stay in WORK for "Output -> New Dataset"
OUT = os.path.join(WORK, "fa15", TAG)
for d in ("specs",):
    shutil.rmtree(os.path.join(SCRATCH, d), ignore_errors=True)
ZIP = shutil.make_archive(os.path.join(WORK, f"fa15_{TAG}_results"), "zip", root_dir=os.path.join(WORK, "fa15"), base_dir=TAG)
print("results archive:", ZIP, f"{os.path.getsize(ZIP) / 1e6:.1f} MB")
for dname in sorted(os.listdir(WORK)):
    if dname.startswith("fa15-asr-"):
        fs = os.listdir(os.path.join(WORK, dname))
        print("dataset folder", dname, len(fs), "files", f"{sum(os.path.getsize(os.path.join(WORK, dname, f)) for f in fs) / 1e9:.2f} GB")
print(open(os.path.join(OUT, "SUMMARY.md")).read())
print("MEASURED notebook runtime (h):", round((time.time() - T_NB) / 3600, 3))
if FAILED:
    raise RuntimeError(f"STOPPED: failed gates {FAILED} (results archived)")